# MCP 101 — a local MCP server + client

Exercise XP: build a tiny MCP server (one tool + one resource) and a Python client that
connects over STDIO, discovers those features, and invokes them.

### What you'll learn
- How MCP structures hosts/clients/servers and why STDIO is perfect for local dev.
- How to register a tool (action) and a resource (read-only context) on a server.
- How to write a client that initializes, lists, and invokes those features.

This notebook installs the `mcp` SDK, writes `server.py` and `client.py` to the Colab VM's
local filesystem with `%%writefile`, then runs the client (which spawns the server itself
over STDIO) so the discovery + invocation output shows up right below the run cell — that
output is what you'd screenshot for submission.

## Setup

Colab already runs inside its own Linux VM, so there's no venv step needed here — just install
the package. (Locally you'd `python -m venv .venv` and activate it first; see this folder's
`README.md` for the local macOS/Linux/Windows setup if you want to run this outside Colab too.)

In [ ]:
!pip install -q "mcp[cli]"
!python --version
!mcp --help

## A. Server (`server.py`)

1. Create an MCP server named `"Demo"`.
2. Add a tool `add(a: int, b: int) -> int` that returns the sum.
3. Add a resource at URI template `greeting://{name}` that returns `"Hello, {name}!"`.
4. Start the server loop over STDIO.

**Note on the import:** the exercise's own scaffold imports
`from mcp.server.fastmcp import FastMCP`. That class was renamed to `MCPServer` (moved to
`mcp.server`) in newer releases of the `mcp` package, with no backward-compatible alias left
behind. The cell below tries the original `FastMCP` import first and falls back to `MCPServer`,
so it works whichever version Colab's `pip install` resolves to — run `!pip show mcp` if you
want to see exactly which version you got.

In [ ]:
%%writefile server.py
"""server.py -- a tiny MCP server exposing one tool and one resource."""

try:
    from mcp.server.fastmcp import FastMCP as MCPServerClass
except ImportError:
    # Newer `mcp` releases renamed FastMCP -> MCPServer and moved it to mcp.server.
    from mcp.server import MCPServer as MCPServerClass

mcp = MCPServerClass("Demo")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Return the sum of two integers."""
    return a + b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    mcp.run()  # transport="stdio" is the default

## B. Client (`client.py`)

1. Use STDIO transport to spawn `server.py` via the `mcp` CLI.
2. Initialize a `ClientSession`.
3. List resources and list tools; print their names.
4. Read `greeting://hello` and call the `add` tool with `a=1, b=7`; print results.

**Note on discovery:** `greeting://{name}` is a *templated* resource (it has a `{name}`
parameter), not a static, ready-to-read one. The MCP protocol keeps those in two separate
lists — `list_resources()` only returns concrete, parameter-free resources, while
`list_resource_templates()` returns the parameterized ones. So `list_resources()` alone will
print `[]` here, which is expected, not a bug; the client below prints both lists so `greet`
actually shows up somewhere in the discovery output.

In [ ]:
%%writefile client.py
"""client.py -- spawns server.py over STDIO, discovers its tools/resources, and invokes them."""

import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


def extract_content(payload):
    """Best-effort to pull text out of an MCP read_resource / call_tool result."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        content = payload.content
        if content:
            first = content[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
        return str(content)
    return str(payload)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # --- List resources and tools ---
            resources_result = await session.list_resources()
            print("Resources:", [r.name for r in resources_result.resources])

            templates_result = await session.list_resource_templates()
            # Attribute name differs across `mcp` versions: snake_case in some,
            # the pydantic alias `resourceTemplates` (camelCase) in others.
            templates = getattr(templates_result, "resource_templates", None)
            if templates is None:
                templates = templates_result.resourceTemplates
            print("Resource templates:", [t.name for t in templates])

            tools_result = await session.list_tools()
            print("Tools:", [t.name for t in tools_result.tools])

            # --- Read the greeting resource ---
            greeting = await session.read_resource("greeting://hello")
            print("greeting://hello ->", extract_content(greeting))

            # --- Call the add tool ---
            sum_result = await session.call_tool("add", {"a": 1, "b": 7})
            print("add(1, 7) ->", extract_content(sum_result))


if __name__ == "__main__":
    asyncio.run(run())

## C. Run

One cell is enough in Colab: `client.py` spawns `server.py` itself over STDIO (the equivalent
of the exercise's "one terminal" mode). The `mcp` console script that pip just installed is
already on this VM's `PATH`, so no venv-activation step is needed here.

If this ever prints a generic `Connection closed` error instead of the expected output, run
`!mcp run server.py` in its own cell first — a server-side exception (like an import error)
will print there directly instead of being swallowed into that message.

In [ ]:
!python client.py

### Expected output

```
Resources: []
Resource templates: ['greet']
Tools: ['add']
greeting://hello -> Hello, hello!
add(1, 7) -> 8
```

`Resources: []` next to `Resource templates: ['greet']` is correct — see the note above Part B.
This is the cell output to screenshot/copy for the "what to submit" terminal capture.

## Troubleshooting

- `mcp: command not found` → re-run the setup cell; outside Colab, activate your venv first.
- No tools/resources visible → check the `@mcp.tool()` / `@mcp.resource()` decorators in
  `server.py`, and try `!mcp run server.py` directly to see any startup error.
- Type mismatch calling `add` → both `a` and `b` need to arrive as JSON integers, not strings.
- `Connection closed` from the client → run `!mcp run server.py` in its own cell; a server-side
  exception (e.g. the `FastMCP`/`MCPServer` import mismatch above) will print there directly.
- `ModuleNotFoundError: No module named 'mcp.server.fastmcp'` → you're on a newer `mcp` release
  that renamed `FastMCP` to `MCPServer`; the try/except import in the server cell above already
  handles this.
- `AttributeError: 'ListResourceTemplatesResult' object has no attribute 'resource_templates'`
  (or the reverse, `resourceTemplates`) → the SDK's field naming for this result has changed
  between `mcp` releases; the `getattr(..., "resource_templates", None)` fallback in the client
  cell above already handles both spellings — confirmed by running this notebook's exact code
  locally against `mcp` 1.28.1, where `resourceTemplates` (camelCase) is the real attribute.